##### Axis 2 with Opphold
**Step 1: Pick only axix one from `https://finnkode.helsedirektoratet.no/phbu/chapter?q=4` and exclude `"2000", "2999", "999", "000"` and date from `1995-01-01`, `save all the axis 2 patient record`` and `axis 2 patient with primary diagnosis`**

In [ ]:
import os
import sys

parent_dir = os.path.join(os.path.dirname(os.path.abspath("")), os.pardir)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
from importall import *

SRC_PARQUET = os.getenv("MAIN_DATASET")
DST_PRIMARY = os.getenv("AXISII_MAIN_DATASET")

df = dd.read_parquet(SRC_PARQUET, engine="pyarrow")
cutoff = "1995-01-01"
df["sak_igangdato"] = dd.to_datetime(df["sak_igangdato"], errors="coerce")
df["sak_avsldato"] = dd.to_datetime(df["sak_avsldato"], errors="coerce")

df = df[(df["sak_igangdato"] >= cutoff) & (df["sak_avsldato"] >= cutoff)]
df = df[(df["patient_age"] >= 1) & (df["patient_age"] <= 19)]
df = df[df["diag_akse"] == "2"]

print(
    "Unique patients • age 1-19 • Axis 2 • episodes ≥ 1995:",
    df["pasient_nr"].nunique().compute(),
)

df_primary = (
    df.fillna({"diag_hoved": "0"})
    .query("diag_hoved == '1'")
    .dropna(subset=["diag_diagnose", "atckode"])
    .loc[lambda d: ~d["diag_diagnose"].isin(["2000", "2999", "999", "000"])]
)
zeros_before = (
    df_primary[df_primary["diag_diagnose"] == "000"]["pasient_nr"].nunique().compute()
)
print(f"Patients with diag_diagnose '000' before exclude: {zeros_before}")


print(
    "Unique patients • all of the above + medicated:",
    df_primary["pasient_nr"].nunique().compute(),
)


DST_PRIMARY.mkdir(parents=True, exist_ok=True)

(
    df_primary.repartition(npartitions=1).to_parquet(
        DST_PRIMARY, engine="pyarrow", write_index=False, write_metadata_file=False
    )
)
print("Parquet files written:")
print(f" {DST_PRIMARY}")

##### Start here
**Step 1(a): Research Question: 1. Overview of BUP data in axis 2 after above filter?**
***Read parquet file with Dask***

In [ ]:
import os
import sys

parent_dir = os.path.join(os.path.dirname(os.path.abspath("")), os.pardir)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
from importall import *
load_dotenv()
output_file19 = os.getenv("AXISII_MAIN_DATASET")
df_axis_one_19 = dd.read_parquet(output_file19, engine="pyarrow")

print("Shape:", df_axis_one_19.shape)
print(
    "Unique cases under 19 with diagnose akse 2 from 1995:",
    df_axis_one_19["sak_nr"].nunique().compute(),
)
print(
    "Unique patients under 19 with diagnose akse 2 from 1995:",
    df_axis_one_19["pasient_nr"].nunique().compute(),
)
print("Oldest sak_igangdato:", df_axis_one_19["sak_igangdato"].min().compute())
print("Newest sak_igangdato:", df_axis_one_19["sak_igangdato"].max().compute())
print("Oldest sak_avsldato:", df_axis_one_19["sak_avsldato"].min().compute())
print("Newest sak_avsldato:", df_axis_one_19["sak_avsldato"].max().compute())
print(
    "Unique diagnose code under 19 with diagnose akse 2 from 1995:",
    df_axis_one_19["diag_diagnose"].nunique().compute(),
)
print(
    "Unique ATC code under 19 with diagnose akse 2 from 1995:",
    df_axis_one_19["atckode"].nunique().compute(),
)
print(
    f"Unique cases under 19 with axis 2 from 1995: {df_axis_one_19["sak_nr"].nunique().compute()} and {df_axis_one_19["opphold_id"].nunique().compute()}"
)

# -------------------------------------------------------------------
# Patient-level age summary for Table 3
# -------------------------------------------------------------------
# patient_age is already calculated upstream from fdt and first episode.
# Therefore, do NOT recalculate age here.
#
# We only collapse repeated analytic rows so that each patient contributes
# once to the demographic summary.

patient_id_col = "pasient_nr"
age_col = "patient_age"

age_by_patient = (
    df_axis_one_19[[patient_id_col, age_col]]
    .assign(
        **{
            age_col: dd.to_numeric(
                df_axis_one_19[age_col],
                errors="coerce"
            )
        }
    )
    .groupby(patient_id_col)[age_col]
    .first()
    .reset_index()
)

age_valid = age_by_patient[age_col].dropna()

(
    total_patients,
    age_nonmissing_n,
    age_missing_n,
    age_mean,
    age_sd,
    age_min,
    age_max,
    age_quantiles,
) = dd.compute(
    age_by_patient[patient_id_col].count(),
    age_valid.count(),
    age_by_patient[age_col].isna().sum(),
    age_valid.mean(),
    age_valid.std(),
    age_valid.min(),
    age_valid.max(),
    age_valid.quantile([0.25, 0.50, 0.75]),
)

age_q1 = age_quantiles.loc[0.25]
age_median = age_quantiles.loc[0.50]
age_q3 = age_quantiles.loc[0.75]

print("\nPatient-level age summary for Table 3")
print("-------------------------------------")
print(f"Age variable: {age_col}")
print(f"Patient identifier: {patient_id_col}")
print(f"Total patients: {total_patients}")
print(f"Non-missing age, n: {age_nonmissing_n}")
print(f"Missing age, n: {age_missing_n}")
print(f"Mean age, years: {age_mean:.1f}")
print(f"SD, years: {age_sd:.1f}")
print(f"Median age, years: {age_median:.1f}")
print(f"IQR, years: {age_q1:.1f}–{age_q3:.1f}")
print(f"Range, years: {age_min:.1f}–{age_max:.1f}")

print(
    "\nSuggested Table 3 wording: "
    f"Age at first episode, years, mean SD: {age_mean:.1f} ({age_sd:.1f}); "
    f"median IQR: {age_median:.1f} ({age_q1:.1f}–{age_q3:.1f}); "
    f"range: {age_min:.1f}–{age_max:.1f}; "
    f"non-missing n={age_nonmissing_n}, missing n={age_missing_n}."
)

**Step 2.1. What is the median contact per patient, range (max and min contact per patient, IQR, top 5% of patient contribute what propotions of contacts) for the patient across Axis 2?**

*To address this comment we did `step 2.1. patient level summary statistics to see median, range, IQR, and find top 5% patient, contact per patient`Regarding Editorial Comment 3 from the previous round, which pointed out that treating individual contacts as independent observations introduces clustering bias, the authors addressed this by adding disclaimers that the study is descriptive and focuses on contact events as the analytic unit. While this transparency is appreciated, relying purely on contact-level data without a patient-level sensitivity analysis leaves open the possibility that the prescribing patterns primarily reflect the habits of a small number of chronic, high-utilizing patients. Authors may wish to provide a brief patient-level summary or sensitivity analysis to validate that the contact-level data is not entirely skewed by outliers.*

- Top 5%: find out who the top 5% of patients are—those with the most contacts—and see how many contacts they contribute. If that small group generates a large chunk of the total (for instance, 40% of all contacts), you note that. This way, you highlight if a few high-contact patients have a disproportionate impact—without redoing everything.

In [ ]:
import os
import sys
import dask
import dask.dataframe as dd

parent_dir = os.path.join(os.path.dirname(os.path.abspath("")), os.pardir)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
from importall import *

load_dotenv()


output_file19 = os.getenv("AXISII_MAIN_DATASET")

df_axis_one_19 = dd.read_parquet(output_file19, engine="pyarrow")

PATIENT_COL = "pasient_nr"
CONTACT_COL = "opphold_id"

print("Shape:", df_axis_one_19.shape)
print("Columns:", df_axis_one_19.columns)

n_unique_cases, n_unique_contacts, n_unique_patients = dask.compute(
    df_axis_one_19["sak_nr"].nunique(),
    df_axis_one_19["opphold_id"].nunique(),
    df_axis_one_19["pasient_nr"].nunique(),
)

print(
    f"Unique cases under 19 with diag_akse 1 from 1995: "
    f"{n_unique_cases} and {n_unique_contacts}"
)

print("Unique patients under 19 with diag_akse 1 from 1995:", n_unique_patients)

# -------------------------------------------------------------------
# Patient-level contact distribution
# -------------------------------------------------------------------
patient_contact_pairs = (
    df_axis_one_19[[PATIENT_COL, CONTACT_COL]].dropna().drop_duplicates()
)

contacts_per_patient_dd = (
    patient_contact_pairs.groupby(PATIENT_COL)[CONTACT_COL].count().rename("contacts")
)

contacts_per_patient = contacts_per_patient_dd.compute().astype("int64")

# -------------------------------------------------------------------
# Summary statistics
# -------------------------------------------------------------------
summary_stats = {
    "number_of_patients": contacts_per_patient.shape[0],
    "total_contacts": contacts_per_patient.sum(),
    "median_contacts_per_patient": contacts_per_patient.median(),
    "minimum_contacts_per_patient": contacts_per_patient.min(),
    "maximum_contacts_per_patient": contacts_per_patient.max(),
    "25th_percentile": contacts_per_patient.quantile(0.25),
    "75th_percentile": contacts_per_patient.quantile(0.75),
    "95th_percentile": contacts_per_patient.quantile(0.95),
}

summary_stats_df = pd.DataFrame(summary_stats.items(), columns=["Statistic", "Value"])

print("\nPatient-level contact distribution summary:")
print(summary_stats_df.to_string(index=False))

# -------------------------------------------------------------------
# Top 5% highest-contact patients
# -------------------------------------------------------------------
n_patients = contacts_per_patient.shape[0]
top_5_percent_n = max(1, math.ceil(n_patients * 0.05))

top_5_percent_patients = contacts_per_patient.sort_values(ascending=False).head(
    top_5_percent_n
)

top_5_contacts = top_5_percent_patients.sum()
total_contacts = contacts_per_patient.sum()
top_5_contact_share = top_5_contacts / total_contacts

print("\nTop 5% highest-contact patients:")
print(f"Number of patients in top 5%: {top_5_percent_n}")
print(f"Contacts contributed by top 5%: {top_5_contacts}")
print(f"Total contacts: {total_contacts}")
print(f"Share of all contacts from top 5%: {top_5_contact_share:.2%}")

if top_5_contact_share >= 0.40:
    print(
        "\nInterpretation: The top 5% of patients contribute at least 40% "
        "of all contacts, indicating a disproportionate impact from a small "
        "high-contact group."
    )
else:
    print(
        "\nInterpretation: The top 5% of patients contribute less than 40% "
        "of all contacts."
    )

**Extra: What are the most frequent medication and diagnosis across all the diagnoses and across the primary or principle diagnosis for the patient across Axis 2?**

In [ ]:
import os
import sys

parent_dir = os.path.join(os.path.dirname(os.path.abspath("")), os.pardir)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
from importall import *

load_dotenv()
from updated_mapped_icd_codes import mapped_icd_codes

output_plot = os.getenv("OUTPUT_PLOTS")

with open(os.getenv("ATCNAME_CODE_NORSK") + "atcnavn_code.json", "r") as f:
    atc_mapping = json.load(f)
atc_map = {code: info.get("atcnavn", "Unknown") for code, info in atc_mapping.items()}
input_path = os.getenv("AXISII_MAIN_DATASET")
df = dd.read_parquet(input_path, engine="pyarrow")

# top 20 diagnosis and medication in axis 1
diag_list = [
    "F810",
    "F813",
    "F83",
    "F801",
    "F82",
    "F819",
    "F802",
    "F809",
    "F812",
    "F811",
    "F800",
    "F808",
    "F818",
    "F80",
    "F89",
    "F88",
    "F81",
]
med_list = [
    "N06BA04",
    "A06BA04",
    "N06BA09",
    "N06BA12",
    "N05CH01",
    "N06AB06",
    "N05AX08",
    "N06AB03",
    "R06AD01",
    "N05AX12",
    "N05AH04",
    "N03AX09",
    "N06BA02",
    "N05CF01",
    "A11EA",
    "N05AA02",
    "D07BC01",
    "N05CD02",
    "N05AH03",
    "H01BA02",
]

diag_counts = (
    df[df["diag_diagnose"].isin(diag_list)]
    .groupby("diag_diagnose")["pasient_nr"]
    .nunique()
    .compute()
    .sort_values(ascending=False)
)

med_counts = (
    df[df["atckode"].isin(med_list)]
    .groupby("atckode")["pasient_nr"]
    .nunique()
    .compute()
    .sort_values(ascending=False)
)

diag_labels = [
    f"{code} – {mapped_icd_codes.get(code,'Unknown')}" for code in diag_counts.index
]
med_labels = [f"{code} – {atc_map.get(code,'Unknown')}" for code in med_counts.index]

wrapped_diag_labels = [textwrap.fill(lbl, width=30) for lbl in diag_labels]
wrapped_med_labels = [textwrap.fill(lbl, width=30) for lbl in med_labels]

# ─── plot diagnoses
width = 0.6
x = np.arange(len(diag_counts))

fig, ax = plt.subplots(figsize=(16, 8))
bars = ax.bar(x, diag_counts.values, width=width, color="#0072B2", alpha=0.6)
ax.set_xticks(x)
ax.set_xticklabels(wrapped_diag_labels, rotation=90)
ax.set_xlabel("Diagnosis (ICD-10 code – description)")
ax.set_ylabel("Unique Patient Count")
ax.set_title("Axis 2 -  Most Common Primary Diagnosis")
ax.bar_label(bars, labels=[f"{v:,}" for v in diag_counts.values], padding=2, fontsize=8)
plt.tight_layout()
# plt.savefig(output_plot + "axis1/axis1_1(b)part1.png", dpi=400, bbox_inches="tight")
plt.show()

# ─── plot medications
x = np.arange(len(med_counts))

fig, ax = plt.subplots(figsize=(16, 8))
bars = ax.bar(x, med_counts.values, width=width, color="#D55E00", alpha=0.6)
ax.set_xticks(x)
ax.set_xticklabels(wrapped_med_labels, rotation=90)
ax.set_xlabel("Medication (ATC code – name)")
ax.set_ylabel("Unique Patient Count")
# ax.set_title("Axis 2 -  Most Common Medication")
ax.bar_label(bars, labels=[f"{v:,}" for v in med_counts.values], padding=2, fontsize=8)
plt.tight_layout()
# plt.savefig(output_plot + "axis1/axis1_1(b)part2.png", dpi=400, bbox_inches="tight")
plt.show()

**Extra: What are the most frequent medication and diagnosis across all the diagnoses and across the primary or principle diagnosis for the `contacts` across Axis 2?**

In [ ]:
import os
import sys

parent_dir = os.path.join(os.path.dirname(os.path.abspath("")), os.pardir)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
from importall import *

load_dotenv()
from updated_mapped_icd_codes import mapped_icd_codes

output_plot = os.getenv("OUTPUT_PLOTS")

with open(os.getenv("ATCNAME_CODE_NORSK") + "atcnavn_code.json", "r") as f:
    atc_mapping = json.load(f)
atc_map = {code: info.get("atcnavn", "Unknown") for code, info in atc_mapping.items()}
input_path = os.getenv("AXISII_MAIN_DATASET")
df = dd.read_parquet(input_path, engine="pyarrow")

# top 20 diagnosis and medication in axis 1
diag_list = [
    "F810",
    "F813",
    "F83",
    "F801",
    "F82",
    "F819",
    "F802",
    "F809",
    "F812",
    "F811",
    "F800",
    "F808",
    "F818",
    "F80",
    "F89",
    "F88",
    "F81",
]
med_list = [
    "N06BA04",
    "A06BA04",
    "N06BA09",
    "N06BA12",
    "N05CH01",
    "N06AB06",
    "N05AX08",
    "N06AB03",
    "R06AD01",
    "N05AX12",
    "N05AH04",
    "N03AX09",
    "N06BA02",
    "N05CF01",
    "A11EA",
    "N05AA02",
    "D07BC01",
    "N05CD02",
    "N05AH03",
    "H01BA02",
]

diag_counts = (
    df[df["diag_diagnose"].isin(diag_list)]
    .groupby("diag_diagnose")["opphold_id"]
    .nunique()
    .compute()
    .sort_values(ascending=False)
)

med_counts = (
    df[df["atckode"].isin(med_list)]
    .groupby("atckode")["opphold_id"]
    .nunique()
    .compute()
    .sort_values(ascending=False)
)

diag_labels = [
    f"{code} – {mapped_icd_codes.get(code,'Unknown')}" for code in diag_counts.index
]
med_labels = [f"{code} – {atc_map.get(code,'Unknown')}" for code in med_counts.index]

wrapped_diag_labels = [textwrap.fill(lbl, width=30) for lbl in diag_labels]
wrapped_med_labels = [textwrap.fill(lbl, width=30) for lbl in med_labels]

# ─── plot diagnoses
width = 0.6
x = np.arange(len(diag_counts))

fig, ax = plt.subplots(figsize=(16, 8))
bars = ax.bar(x, diag_counts.values, width=width, color="#0072B2", alpha=0.6)
ax.set_xticks(x)
ax.set_xticklabels(wrapped_diag_labels, rotation=90)
ax.set_xlabel("Diagnosis (ICD-10 code – description)")
ax.set_ylabel("Unique Contact Count")
ax.set_title("Axis 2 -  Most Common Primary Diagnosis")
ax.bar_label(bars, labels=[f"{v:,}" for v in diag_counts.values], padding=2, fontsize=8)
plt.tight_layout()
# plt.savefig(output_plot + "axis1/axis1_1(b)part1.png", dpi=400, bbox_inches="tight")
plt.show()

# ─── plot medications
x = np.arange(len(med_counts))

fig, ax = plt.subplots(figsize=(16, 8))
bars = ax.bar(x, med_counts.values, width=width, color="#D55E00", alpha=0.6)
ax.set_xticks(x)
ax.set_xticklabels(wrapped_med_labels, rotation=90)
ax.set_xlabel("Medication (ATC code – name)")
ax.set_ylabel("Unique Contact Count")
# ax.set_title("Axis 2 -  Most Common Medication")
ax.bar_label(bars, labels=[f"{v:,}" for v in med_counts.values], padding=2, fontsize=8)
plt.tight_layout()
# plt.savefig(output_plot + "axis1/axis1_1(b)part2.png", dpi=400, bbox_inches="tight")
plt.show()